In [39]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [40]:
%sql sqlite:///:memory:

In [41]:
%sql sqlite:///my_database.db

In [ ]:
# Select top purchasing customer for each store based on total purchase amount
# columns: store_id, customer_id, purchase_amt
# tbl: sales
# Input:

store_id, customer_id, purchase_Amt
1, 101, 500
1, 101, 300
1, 102, 500
2, 101, 1000
2, 102, 900
3, 101, 1000
3, 102, 1000


In [6]:
%%sql
CREATE TABLE sales (
store_id INT,
customer_id INT,
purchase_Amt INT
);

insert into sales (store_id, customer_id, purchase_Amt) 
values(1, 101, 500),
(1, 101, 300),
(1, 102, 500),
(2, 101, 1000),
(2, 102, 900),
(3, 101, 1000),
(3, 102, 1000);


   sqlite:///:memory:
 * sqlite:///my_database.db
(sqlite3.OperationalError) table sales already exists
[SQL: CREATE TABLE sales (
store_id INT,
customer_id INT,
purchase_Amt INT
);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [4]:
%%sql 
SELECT * FROM sales;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


store_id,customer_id,purchase_Amt
1,101,500
1,101,300
1,102,500
2,101,1000
2,102,900
3,101,1000
3,102,1000


In [13]:
%%sql
select store_id, customer_id, total_purchase from (
select store_id, customer_id, total_purchase, DENSE_RANK() OVER(PARTITION BY store_id order by total_purchase desc) as rank from
(select store_id, customer_id, SUM(purchase_Amt) as total_purchase from sales group by 1,2))
where rank=1;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


store_id,customer_id,total_purchase
1,101,800
2,101,1000
3,101,1000
3,102,1000


##create pivot table from below table

In [ ]:
##create pivot table from below table
Product		Comp   		Price
iPhone		Amazon		100
iPhone		Flipkart   	90
iPhone		Snapdeal	110
iPhone 11	Amazon		100
iPhone 11	Flipkart   	120
iPhone 11	Snapdeal	110

#expected output
Product   	Amz_price	Flipkart_price	Snap_price
iPhone		100			90				110		
iPhone 11   100			120				110

In [14]:
%%sql
CREATE TABLE ProductPrice (
    Product VARCHAR(50),
    Comp VARCHAR(50),
    Price INT
);
INSERT INTO ProductPrice (Product, Comp, Price) VALUES
('iPhone', 'Amazon', 100),
('iPhone', 'Flipkart', 90),
('iPhone', 'Snapdeal', 110),
('iPhone 11', 'Amazon', 100),
('iPhone 11', 'Flipkart', 120),
('iPhone 11', 'Snapdeal', 110);

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.
6 rows affected.


[]

In [29]:
%%sql
select product, 
MAX(CASE WHEN Comp='Amazon' then price else null end) as amazon_price,
MAX(CASE WHEN Comp='Flipkart' then price else null end) as flipkart_price,
MAX(CASE WHEN Comp='Snapdeal' then price else null end) as snapdeal_price
 from ProductPrice
 group by 1;


   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


Product,amazon_price,flipkart_price,snapdeal_price
iPhone,100,90,110
iPhone 11,100,120,110


In [ ]:
## below are two tables
## what is output of below query

sum(a),sum(b),sum(a+b),sum(a)+sum(b)


a	b
1	null
null 1
1	 1

In [30]:
%%sql
CREATE TABLE t1 (a,b);

insert into t1 (a, b) 
values(1,null ),
(null, 1),
(1, 1);

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.
3 rows affected.


[]

In [34]:
%%sql
select * from t1;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


a,b
1,None
None,1
1,1


In [35]:
%%sql
select sum(a),sum(b),sum(a+b),sum(a)+sum(b) from t1;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


sum(a),sum(b),sum(a+b),sum(a)+sum(b)
2,2,2,4


#### Join results of all joins

In [ ]:
t1  
1
1
2
3
5
Null

t2
1
1
2
4
Null
Null

In [ ]:
%%sql
# CREATE TABLE t2 (a);
insert into t2 (a) 
values(1),
(1),
(2),
(3),
(5),
(NULL);

In [44]:
%%sql
CREATE TABLE t3 (a);
insert into t3 (a) 
values(1),
(1),
(2),
(4),
(NULL),
(NULL);

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.
6 rows affected.


[]

In [49]:
%%sql
select * from t2 left join t3
on t2.a=t3.a;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


a,a_1
1,1
1,1
1,1
1,1
2,2
3,None
5,None
None,None


In [51]:
%%sql
select * from t2 inner join t3
on t2.a=t3.a;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


a,a_1
1,1
1,1
1,1
1,1
2,2


In [54]:
%%sql
select * from t3 left join t2
on t2.a=t3.a;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


a,a_1
1,1
1,1
1,1
1,1
2,2
4,None
None,None
None,None


In [55]:
%%sql
select * from t2 full outer join t3
on t2.a=t3.a;

   sqlite:///:memory:
 * sqlite:///my_database.db
(sqlite3.OperationalError) RIGHT and FULL OUTER JOINs are not currently supported
[SQL: select * from t2 full outer join t3
on t2.a=t3.a;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [ ]:
# write query to find net amount per customer ID from below table  
# subtract for debit and add for credit
Id item amount category 
1 a 10 debit 
1 a 20 credit 
1 a 24 debit 
2 b 5 debit 
2 b 50 credit

In [56]:
%%sql
CREATE TABLE transactions (
    id INT,
    item VARCHAR(50),
    amount INT,
    category VARCHAR(20)
);
INSERT INTO transactions (id, item, amount, category) VALUES
(1, 'a', 10, 'debit'),
(1, 'a', 20, 'credit'),
(1, 'a', 24, 'debit'),
(2, 'b', 5, 'debit'),
(2, 'b', 50, 'credit');


   sqlite:///:memory:
 * sqlite:///my_database.db
Done.
5 rows affected.


[]

In [57]:
%%sql
select item, 
SUM(CASE when category ='debit' then amount*-1
    when category ='credit' then amount
    ELSE 0
    END) net_amount
from transactions
group by item;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


item,net_amount
a,-14
b,45


In [85]:
%%sql
drop table cust_transactions;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


[]

In [ ]:
%%sql
select CAST(tx_date as DATE) as dates from cust_transactions;

In [ ]:
#write sql query for pivot
# input table
Product Year Revenue 
A 2021 100 
A 2022 150 
B 2021 200 
B 2022 250

# output table
Product 2021 2022 
A 100 150 
B 200 250

In [92]:
%%sql
select product,
SUM(CASE when year=2021 then revenue END) as y2021,
SUM(CASE when year=2022 then revenue END) as y2022
from product_rev
group by 1;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


Product,y2021,y2022
A,100,150
B,200,250


In [94]:
%%sql
select product, year, SUM(revenue) from product_rev
 group by product , year


   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


Product,Year,SUM(revenue)
A,2021,100
A,2022,150
B,2021,200
B,2022,250


In [ ]:
%%sql
-- create example table
CREATE TABLE emp_salary (
    emp_id INTEGER,
    department TEXT,
    salary INT
);

-- sample data (includes ties and a dept with only one employee)
INSERT INTO emp_salary (emp_id, department, salary) VALUES
(1, 'HR', 5000),
(2, 'HR', 7000),
(3, 'HR', 7000),
(4, 'HR', 4000),
(5, 'Eng', 12000),
(6, 'Eng', 11000),
(7, 'Eng', 12000),
(8, 'Eng', 9000),
(9, 'Sales', 8000),
(10,'Sales', 8000),
(11,'Sales', 7000),
(12,'Legal', 6000);

In [97]:
%%sql
SELECT department, salary AS second_highest_salary
FROM (
    SELECT department, salary,
                 DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS rnk
    FROM emp_salary
) t
WHERE rnk = 2
ORDER BY department;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


department,second_highest_salary
Eng,11000
HR,5000
Sales,7000


In [101]:
%%sql
select a, COALESCE(a, 0) from t1;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


a,"COALESCE(a, 0)"
1,1
None,0
1,1
1,1
1,1
2,2
3,3
5,5
None,0


## what will be output for below tables with joins
Table A
column_a
1
1
1
1
 
Table B     
column_b                            
1
1


left join:  8 rows
right join 8 rows
inner join 8 rows
ourter join 8 rows

In [ ]:
%%sql
# create table A (a int);
# create table B (b int);
# insert into A (a) values(1),(1),(1),(1);
# insert into B (b) values(1),(1);


In [116]:
%%sql
select count(*) from A left join B on A.a=B.b;


   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


count(*)
8


In [115]:
%%sql
select count(*) from A inner join B on A.a=B.b;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


count(*)
8


In [117]:
%%sql
CREATE TABLE Test1 (
  Col1 INT
);
INSERT INTO Test1 (Col1) VALUES
(1),
(1),
(1),
(1);
CREATE TABLE Test2 (
  Col1 INT
);

INSERT INTO Test2 (Col1) VALUES
(1),
(1),
(1),
(1),
(NULL);

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.
4 rows affected.
Done.
5 rows affected.


[]

In [119]:
%%sql
SELECT COUNT (*) FROM TEST1 CROSS JOIN TEST2; 

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


COUNT (*)
20


In [120]:
%%sql
SELECT COUNT (*) FROM TEST1 A, TEST2 B WHERE A.COL1 = B.COL1; 


   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


COUNT (*)
16


In [121]:
%%sql
SELECT COUNT (*) FROM TEST2 A LEFT OUTER JOIN TEST1 B ON A.COL1 = B.COL1; 

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


COUNT (*)
17


## Find the store having maximum customers

In [ ]:
%%sql
CREATE TABLE customer (
    id INT,
    name VARCHAR(50),
    address VARCHAR(20)
);
INSERT INTO customer (id, name, address) VALUES
(1,'A',	'X'),
(2,'B',	'Y'),
(3,'C',	'Z'),
(1,'D',	'X');

CREATE TABLE store (
    id INT,
    name VARCHAR(50),
    location VARCHAR(20)
);
INSERT INTO store (id, name, location) VALUES
(1,'P',	'X'),
(2,'Q',	'W'),
(3,'R',	'Z');

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.
3 rows affected.
Done.
3 rows affected.


[]

In [125]:
%%sql
SELECT 
    s.id AS store_id,
    s.name AS store_name,
    COUNT(c.id) AS customer_count
FROM store s
LEFT JOIN customer c 
    ON s.location = c.address
GROUP BY s.id, s.name
ORDER BY customer_count DESC
LIMIT 1;

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


store_id,store_name,customer_count
1,P,2


## sql to get the address for which store not available

In [126]:
%%sql
select distinct address from customer where Address not in (select distinct location from store);

   sqlite:///:memory:
 * sqlite:///my_database.db
Done.


address
Y
